In [ ]:
import pandas as pd
import yfinance as yf

### Question 1: Which year had the highest number of additions (starting from 2020)?

In [ ]:
import requests
from io import StringIO
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
response = requests.get(url, headers=headers)
df = pd.read_html(StringIO(response.text))
df = df[0]
df['Date added'] = pd.to_datetime(df['Date added'])
df['Date added'] = df['Date added'].dt.year
df

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011,1467373,1989
...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011,1524472,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997,1041061,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019,877212,1969
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001,1136869,1927


In [ ]:
df_filtered = df[(df['Date added'] >= 2020) & (df['Date added'] < 2026)]
df_filtered.groupby('Date added')['Date added'].count()

,Date added
Date added,
2020,10
2021,10
2022,15
2023,15
2024,16
2025,18


most companies were added in 2025

In [ ]:
import datetime

x = datetime.datetime.now()

current_year = x.year
df[df['Date added'] < 2006]['Symbol'].count()

np.int64(218)

218 companies have been on the index for more than 20 years

## Question 2: How many indexes (out of 10) have better year-to-date returns than the US (S&P 500) as of August 21, 2026?

In [ ]:
import yfinance as yf

sp = yf.Ticker("^GSPC")
sp = sp.history(start='2026-01-01', end='2026-08-21')
sp = sp.reset_index()
start_date = x.iloc[0]['Close']
current_date = x.iloc[-1]['Close']
sp_ytd_rate = (current_date - start_date) / start_date * 100

count = 0
ticker_list = ['000001.SS', '^HSI', '^AXJO', '^NSEI', '^GSPTSE', '^GDAXI', '^FTSE', '^N225', '^MXX', '^BVSP']
for ticker in ticker_list:
  x = yf.Ticker(ticker)
  x = x.history(start='2026-01-01', end='2026-08-21')
  x = x.reset_index()
  start_date = x.iloc[0]['Close']
  current_date = x.iloc[-1]['Close']
  ytd_rate = (current_date - start_date) / start_date * 100

  if ytd_rate > sp_ytd_rate:
    count += 1
    print(ticker, ytd_rate)
print(f"Number of indexes with better YTD than S&P 500: {count}")

^GSPTSE 14.057465615219911
^GDAXI 5.88320316659904
^FTSE 8.010175932582827
^N225 27.750744826533207
Number of indexes with better YTD than S&P 500: 4


### Question 3: Calculate the median drawdown (in %) of significant market corrections in the S&P 500 index.

In [ ]:
import yfinance as yf

indexes = []
df = yf.Ticker("^GSPC")
df = df.history(start='1950-01-01')
df = df.reset_index()
df['ath'] = df['Close'].cummax()
for index, row in df.iterrows():
  if row['Close'] == row['ath']:
    indexes.append(index)
df

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,ath
0,1950-01-03 00:00:00-05:00,16.660000,16.660000,16.660000,16.660000,1260000,0.0,0.0,16.660000
1,1950-01-04 00:00:00-05:00,16.850000,16.850000,16.850000,16.850000,1890000,0.0,0.0,16.850000
2,1950-01-05 00:00:00-05:00,16.930000,16.930000,16.930000,16.930000,2550000,0.0,0.0,16.930000
3,1950-01-06 00:00:00-05:00,16.980000,16.980000,16.980000,16.980000,2010000,0.0,0.0,16.980000
4,1950-01-09 00:00:00-05:00,17.080000,17.080000,17.080000,17.080000,2520000,0.0,0.0,17.080000
...,...,...,...,...,...,...,...,...,...
19285,2026-08-28 00:00:00-04:00,7735.169922,7771.479980,7700.910156,7711.759766,4297560000,0.0,0.0,7798.990234
19286,2026-08-31 00:00:00-04:00,7697.520020,7697.520020,7665.060059,7686.140137,5386090000,0.0,0.0,7798.990234
19287,2026-09-01 00:00:00-04:00,7635.470215,7663.629883,7611.200195,7631.470215,4858180000,0.0,0.0,7798.990234
19288,2026-09-02 00:00:00-04:00,7634.580078,7681.189941,7633.620117,7666.600098,4738910000,0.0,0.0,7798.990234


In [ ]:
corrections = []
for i in range(len(indexes) - 1):
  start = indexes[i]
  end = indexes[i+1]
  j = start + 1
  minimum = df.loc[start, 'Close']
  minimum_index = start

  while j < end :
    if df.loc[j, 'Close'] < minimum:
      minimum = df.loc[j, 'Close']
      minimum_index = j
    j += 1
  drawdown = (df.loc[start, 'Close']- minimum) /df.loc[start, 'Close'] *100
  if drawdown >= 5:
    #duration = minimum_index - start
    duration = (df.loc[minimum_index, 'Date'] - df.loc[start, 'Date']).days
    #print(start, end, minimum, drawdown, duration)
    corrections.append([drawdown, duration])
corrections = pd.DataFrame(corrections, columns=['drawdown', 'duration'])
corrections['drawdown'].median()
#corrections.sort_values(by='duration', ascending=False)

7.986357561745869

The median significant drawdown (over 5%) is around 8%

###Question 4: Calculate the median 2-day percentage change in stock prices following positive earnings surprise days.

In [ ]:
import yfinance as yf
import pandas as pd

amzn = yf.Ticker('AMZN')
amzn_earnings = amzn.get_earnings_dates()
amzn_stock_prices = amzn.history(start='2020-10-01')
amzn_stock_prices.reset_index(inplace=True)
amzn_earnings.reset_index(inplace=True)
amzn_earnings = amzn_earnings.rename(columns={'Earnings Date': 'Date'})

amzn_earnings['Date'] = amzn_earnings['Date'].dt.date
amzn_stock_prices['Date'] = amzn_stock_prices['Date'].dt.date

start = 0
end = start + 2
for i in range(len(amzn_stock_prices)-2):
  percentage = (amzn_stock_prices.iloc[i+2]['Close']/amzn_stock_prices.iloc[i]['Close'] - 1)*100
  amzn_stock_prices.loc[i+1, 'percentages'] = percentage


merged_df = pd.merge(amzn_earnings, amzn_stock_prices, on='Date', how='inner')
merged_df = merged_df[merged_df['Surprise(%)'] > 0]
merged_df['percentages'].median()

0.35280649406906894

In [ ]:
merged_df[['Surprise(%)', 'percentages']].corr()

,Surprise(%),percentages
Surprise(%),1.000000,0.330645
percentages,0.330645,1.000000
